In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 📝 Human-in-the-Loop Contract Reviewer

## What We're Building

A contract review workflow where the LLM:
1. **Drafts review comments** — flags risky clauses and issues
2. **Pauses for human approval** — human reviews the draft
3. **Generates final revision notes** — incorporates feedback

## Why Human-in-the-Loop (HITL)?

```
Without HITL:  LLM generates → ships to client  →  Mistakes catch you off-guard
With HITL:     LLM generates → ⏸ PAUSE → human reviews → finalize  →  Safe + fast
```

## LangGraph HITL Pattern

```
┌─────────────┐     ┌──────────────┐     ┌────────────┐
│ Analyze      │ ──▶ │ Draft Review │ ──▶ │ ⏸ PAUSE    │
│ Contract     │     │ Comments     │     │ for Human  │
└─────────────┘     └──────────────┘     └─────┬──────┘
                                                │
                                      Human approves/edits
                                                │
                                         ┌──────▼──────┐
                                         │ Generate     │
                                         │ Final Notes  │
                                         └─────────────┘
```

## Key LangGraph Concept: `interrupt_before`

When you compile a graph with `interrupt_before=["node_name"]`, the graph
**pauses execution** before that node runs. The human can inspect the state,
modify it, and then resume. This requires a **checkpointer** (MemorySaver)
to save/restore state.

## Stack
- **LangGraph** — workflow orchestration with HITL
- **Ollama** — local LLM (qwen3.5:9b)

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.2)
print("All imports successful!")

## Step 2 — Define State & Sample Contract

In [ ]:
class ContractReviewState(TypedDict):
    """State that flows through the contract review workflow."""
    contract_text: str            # Raw contract text
    clause_analysis: str          # LLM's clause-by-clause analysis
    draft_comments: str           # Draft review comments
    human_feedback: str           # Human's edits/approval (filled at pause)
    final_revision_notes: str     # Final output after feedback


# Sample contract for review
sample_contract = """
SERVICE AGREEMENT

1. TERM: This agreement is effective for 36 months from the signing date.
   Early termination requires 180 days written notice AND payment of a
   termination fee equal to 50% of remaining contract value.

2. PAYMENT: Client shall pay $15,000/month. Late payments incur 18% annual
   interest compounding daily. Provider may suspend services after 5 days
   of non-payment without prior notice.

3. LIABILITY: Provider's total liability is capped at fees paid in the
   most recent 1-month period. Provider is not liable for any indirect,
   consequential, or punitive damages regardless of cause.

4. IP RIGHTS: All work product created during the engagement becomes the
   exclusive property of the Provider. Client receives a non-exclusive,
   non-transferable license to use deliverables.

5. DATA: Provider may retain copies of all client data for up to 7 years
   after termination for compliance purposes. Provider may use anonymized
   client data for benchmarking and marketing.

6. INDEMNIFICATION: Client shall indemnify Provider against all claims,
   damages, and expenses arising from Client's use of the services,
   including Provider's own negligence.

7. GOVERNING LAW: This agreement is governed by the laws of the Cayman
   Islands. All disputes must be resolved through binding arbitration
   in the Cayman Islands.
"""

print("Sample contract loaded")
print(f"Length: {len(sample_contract)} characters, {len(sample_contract.split())} words")

## Step 3 — Build the Graph Nodes

In [ ]:
# --- Node 1: Analyze the contract clause by clause ---
analyze_prompt = ChatPromptTemplate.from_template(
    """You are a contract analyst. Analyze each clause in this contract and
identify potential risks, unusual terms, or one-sided provisions.

For each clause, provide:
- Risk Level: LOW / MEDIUM / HIGH / CRITICAL
- Issue: What's concerning
- Standard Practice: What's normal in the industry

Contract:
{contract}

Clause-by-clause analysis:"""
)


def analyze_contract(state: ContractReviewState) -> dict:
    """Analyze each clause for risks."""
    print("📋 Node: analyze_contract — Scanning clauses...")
    chain = analyze_prompt | llm | StrOutputParser()
    analysis = chain.invoke({"contract": state["contract_text"]})
    return {"clause_analysis": analysis}


# --- Node 2: Draft review comments ---
draft_prompt = ChatPromptTemplate.from_template(
    """Based on this clause analysis, draft review comments for the client.
Write as a legal advisor. Be specific about what to negotiate or push back on.

Format as numbered comments, each with:
- The clause reference
- The issue
- Your recommendation

Analysis:
{analysis}

Draft review comments:"""
)


def draft_review_comments(state: ContractReviewState) -> dict:
    """Generate draft review comments for human approval."""
    print("✏️ Node: draft_review_comments — Writing comments...")
    chain = draft_prompt | llm | StrOutputParser()
    comments = chain.invoke({"analysis": state["clause_analysis"]})
    return {"draft_comments": comments}


# --- Node 3: Generate final revision notes (after human feedback) ---
finalize_prompt = ChatPromptTemplate.from_template(
    """You are preparing final revision notes for a contract review.

Original draft comments:
{draft_comments}

Human reviewer's feedback:
{feedback}

Generate polished FINAL REVISION NOTES incorporating the feedback.
These notes will be sent to the counterparty's legal team.
Be professional and specific.

Final revision notes:"""
)


def generate_final_notes(state: ContractReviewState) -> dict:
    """Produce final revision notes incorporating human feedback."""
    print("📄 Node: generate_final_notes — Finalizing...")
    chain = finalize_prompt | llm | StrOutputParser()
    notes = chain.invoke({
        "draft_comments": state["draft_comments"],
        "feedback": state["human_feedback"]
    })
    return {"final_revision_notes": notes}


print("Three workflow nodes defined")

## Step 4 — Build & Compile the Graph with HITL

The key: `interrupt_before=["generate_final_notes"]` pauses the graph
**after** draft comments are ready but **before** final notes are generated,
giving the human a chance to review and provide feedback.

In [ ]:
# Build the graph
workflow = StateGraph(ContractReviewState)

# Add nodes
workflow.add_node("analyze_contract", analyze_contract)
workflow.add_node("draft_review_comments", draft_review_comments)
workflow.add_node("generate_final_notes", generate_final_notes)

# Connect edges: linear pipeline
workflow.set_entry_point("analyze_contract")
workflow.add_edge("analyze_contract", "draft_review_comments")
workflow.add_edge("draft_review_comments", "generate_final_notes")
workflow.add_edge("generate_final_notes", END)

# Compile WITH checkpointer and interrupt
memory = MemorySaver()
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["generate_final_notes"]  # ⏸ Pause here for human review
)

print("Graph compiled with HITL pause before 'generate_final_notes'")

## Step 5 — Run Phase 1: LLM Analyzes & Drafts

The graph runs until it hits the interrupt point.

In [ ]:
# Thread config — required for checkpointing
config = {"configurable": {"thread_id": "contract-review-1"}}

# Start the workflow
initial_state = {
    "contract_text": sample_contract,
    "clause_analysis": "",
    "draft_comments": "",
    "human_feedback": "",
    "final_revision_notes": ""
}

print("🚀 Starting contract review workflow...\n")
print("The graph will pause BEFORE generating final notes.\n")

# This will run analyze_contract → draft_review_comments → PAUSE
result = app.invoke(initial_state, config)

print("\n" + "="*60)
print("⏸️ WORKFLOW PAUSED — Waiting for human review")
print("="*60)
print("\n📝 Draft comments for your review:")
print(result["draft_comments"])

## Step 6 — Human Reviews & Provides Feedback

In a real app, this would be a UI form or chat input.
Here we simulate the human providing feedback.

In [ ]:
# Simulate human feedback
human_feedback = """
Good analysis overall. My additional notes:
1. The IP clause (Clause 4) is the TOP PRIORITY — we absolutely must own the work product.
   Negotiate this to "all deliverables belong to Client." This is a deal-breaker.
2. The Cayman Islands jurisdiction is suspicious. Push for our home jurisdiction.
3. The termination fee is too high but we can negotiate to 25% instead of 50%.
4. Keep the indemnification comment — the "including Provider's own negligence" part
   is unacceptable and likely unenforceable anyway.
5. Tone: be firm but professional. We want this deal to work.
"""

print("👤 Human feedback provided:")
print(human_feedback)

## Step 7 — Resume the Graph with Human Feedback

We update the state with the human's feedback and resume execution.
The graph picks up right where it paused (before `generate_final_notes`).

In [ ]:
# Update the saved state with human feedback
app.update_state(config, {"human_feedback": human_feedback})

print("🔄 Resuming workflow with human feedback...\n")

# Resume — this runs generate_final_notes → END
final_result = app.invoke(None, config)

print("\n" + "="*60)
print("✅ WORKFLOW COMPLETE — Final Revision Notes")
print("="*60)
print(final_result["final_revision_notes"])

## Step 8 — Inspect the Full State History

In [ ]:
# View the state at each checkpoint
print("📜 State History (checkpoints):\n")
for i, snapshot in enumerate(app.get_state_history(config)):
    node = snapshot.metadata.get("source", "unknown") if snapshot.metadata else "init"
    keys_filled = [k for k, v in snapshot.values.items() if v]
    print(f"  Checkpoint {i}: source={node}, filled_keys={keys_filled}")

## 🧠 Key Concepts Recap

| LangGraph Feature | What It Does | Used Here |
|-------------------|--------------|-----------|
| `StateGraph` | Define a typed state that flows through nodes | `ContractReviewState` |
| `MemorySaver` | In-memory checkpointer to save/restore state | Required for HITL |
| `interrupt_before` | Pause the graph before a specific node | Before `generate_final_notes` |
| `update_state()` | Modify saved state (inject human input) | Add feedback to state |
| `invoke(None, config)` | Resume from the last checkpoint | Continue after pause |
| `thread_id` | Identifies a conversation/workflow instance | Track separate reviews |

## HITL Flow Summary

```python
# Phase 1: Run until pause
result = app.invoke(initial_state, config)       # Runs → hits interrupt → pauses

# Phase 2: Human reviews result["draft_comments"]
app.update_state(config, {"human_feedback": ...})  # Inject human input

# Phase 3: Resume
final = app.invoke(None, config)                 # Continues from pause point
```

## 🔧 Extensions

- **Multiple pause points**: Add `interrupt_before` for several nodes
- **Rejection loop**: If human rejects, loop back to re-draft
- **Web UI**: Connect to a Streamlit/Gradio interface for the review step
- **Email integration**: Pause and send email notification, resume on reply